# 07 - IOHanalyzer Benchmarking & Comparison (Plotly PNG Export)

This notebook:
1. Loads performance trajectory logs (`.dat` files) from `data/ioh_logs/` for target BBOB problems across multiple dimensions (`DIMS`) and noise levels (`NOISE_STDS`).
2. Compares **LLaMEA Champion** against classical baselines (**CMA-ES**, **DE**, **PSO**).
3. Computes:
   - **Fixed-Budget Convergence Curves** (Mean ground-truth clean error vs. Function Evaluations).
   - **Empirical Cumulative Distribution Functions (ECDF)** over target error thresholds.
4. Exports high-resolution Plotly PNG figures to `figures/{dim}D/std_{noise_std}/f{p_id}.png` without showing inline.


In [4]:
import os
import re
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

cwd = Path('.').resolve()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
IOH_LOGS_DIR = PROJECT_ROOT / 'data' / 'ioh_logs'
FIGURES_DIR  = PROJECT_ROOT / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ── Benchmark Configuration ──────────────────────────────
TARGET_PROBLEMS = [1, 8, 11, 15, 21]
DIMS            = [2, 3]             # Dimensions to evaluate
NOISE_STDS      = [0.0, 0.05, 0.1]  # Noise levels to evaluate
# ─────────────────────────────────────────────────────────

ALGO_COLORS = {
    'LLaMEA Champion': '#d62728',  # Vibrant Red
    'LLaMEA': '#d62728',          # Vibrant Red
    'CMA-ES': '#1f77b4',          # Vibrant Blue
    'DE': '#2ca02c',              # Emerald Green
    'PSO': '#ff7f0e'               # Warm Orange
}

def hex_to_rgba(hex_color: str, alpha: float = 0.15) -> str:
    """Convert hex color string to rgba CSS string."""
    hex_color = hex_color.lstrip('#')
    r = int(hex_color[0:2], 16)
    g = int(hex_color[2:4], 16)
    b = int(hex_color[4:6], 16)
    return f'rgba({r}, {g}, {b}, {alpha})'

print(f'IOH Logs Path: {IOH_LOGS_DIR}')
print(f'Figures Export Path: {FIGURES_DIR}')
print(f'Dimensions: {DIMS}')
print(f'Noise STDs: {NOISE_STDS}')


IOH Logs Path: /Users/nicolaibrahim/Desktop/proj/AAD_LLM/data/ioh_logs
Figures Export Path: /Users/nicolaibrahim/Desktop/proj/AAD_LLM/figures
Dimensions: [2, 3]
Noise STDs: [0.0, 0.05, 0.1]


## 1. Load IOH Trajectory Data

In [5]:
import io

def parse_ioh_dat_file(dat_path: Path) -> list[pd.DataFrame]:
    """Parse a single (possibly multi-run) IOH .dat file into a list of run DataFrames."""
    if not dat_path.exists() or dat_path.stat().st_size == 0:
        return []
        
    text = dat_path.read_text(encoding='utf-8', errors='ignore').strip()
    if not text:
        return []
        
    runs = []
    # Split text on occurrences of header line (or start of file)
    blocks = [b.strip() for b in text.split('evaluations') if b.strip()]
    
    for block in blocks:
        lines = [l for l in block.splitlines() if l.strip() and not l.strip().startswith('#') and not l.strip().startswith('raw_y')]
        if not lines:
            continue
        csv_data = 'evaluations raw_y' + chr(10) + chr(10).join(lines)
        try:
            df = pd.read_csv(io.StringIO(csv_data), sep=r'\s+')
            df['evaluations'] = pd.to_numeric(df['evaluations'], errors='coerce')
            df['raw_y'] = pd.to_numeric(df['raw_y'], errors='coerce')
            df = df.dropna().reset_index(drop=True)
            if len(df) > 0:
                runs.append(df)
        except Exception:
            pass
            
    return runs


def load_ioh_dat_files(problem_dir: Path, ignore_test_dummies: bool = True):
    """Load all .dat files grouped by algorithm from a problem directory."""
    data = {}
    if not problem_dir.exists():
        return data
        
    for algo_dir in sorted(problem_dir.iterdir()):
        if not algo_dir.is_dir():
            continue
            
        # Strip trailing numeric suffixes (e.g. -1, -2) if present
        base_name = re.sub(r'-\d+$', '', algo_dir.name)
        clean_name = base_name.split('_std')[0]
        
        # Ignore temporary artifact directories generated during unit tests
        if ignore_test_dummies and any(dk in clean_name for dk in ["dummy-llm", "failing-llm", "local-model"]):
            continue
            
        if 'llamea_champion' in clean_name or 'champion' in clean_name:
            display_name = 'LLaMEA Champion'
        elif 'llamea' in clean_name or 'qwen' in clean_name or 'llama' in clean_name or 'claude' in clean_name or 'gpt' in clean_name:
            clean_model = clean_name.replace("llamea_", "").split("_exp")[0].removesuffix(".gguf").removesuffix(".bin").replace("-instruct", "").replace("_q4_k_m", "")
            display_name = f'LLaMEA ({clean_model})'
        else:
            display_name = clean_name.upper()
            
        runs = []
        for dat_file in algo_dir.rglob('*.dat'):
            parsed_runs = parse_ioh_dat_file(dat_file)
            runs.extend(parsed_runs)
                
        if runs:
            if display_name in data:
                data[display_name].extend(runs)
            else:
                data[display_name] = runs
                
    return data

print('Data loading helper defined.')


Data loading helper defined.


## 2. Fixed-Budget Convergence & ECDF Plots

In [6]:
for dim in DIMS:
    for noise_std in NOISE_STDS:
        for p_id in TARGET_PROBLEMS:
            prob_dir = IOH_LOGS_DIR / f"{dim}D" / f"std_{noise_std}" / f"f{p_id}"
            algo_runs = load_ioh_dat_files(prob_dir)
            
            if not algo_runs:
                print(f'f{p_id} ({dim}D, noise={noise_std}): No IOH log files found at {prob_dir}. Run Notebooks 05 and 06 first.')
                continue
                
            fig = make_subplots(
                rows=1, cols=2,
                subplot_titles=(
                    f'BBOB f{p_id} ({dim}D, Noise std={noise_std}) - Convergence',
                    f'BBOB f{p_id} ({dim}D, Noise std={noise_std}) - ECDF'
                ),
                horizontal_spacing=0.10
            )
            
            # --- Subplot 1: Convergence (Clean Error vs Evaluations) ---
            for algo_name, runs in algo_runs.items():
                color = ALGO_COLORS.get(algo_name, '#7f7f7f')
                rgba_color = hex_to_rgba(color, 0.15)
                
                eval_grid = np.logspace(0, 5, 200)
                interp_y = []
                for df in runs:
                    evals = df['evaluations'].values
                    errors = df['raw_y'].values
                    cum_min_errors = np.minimum.accumulate(errors)
                    y_interp = np.interp(eval_grid, evals, cum_min_errors, left=cum_min_errors[0], right=cum_min_errors[-1])
                    interp_y.append(y_interp)
                    
                mean_y = np.mean(interp_y, axis=0)
                std_y = np.std(interp_y, axis=0)
                upper_y = mean_y + std_y
                lower_y = np.maximum(mean_y - std_y, 1e-12)
                
                # Upper bound (invisible line)
                fig.add_trace(
                    go.Scatter(
                        x=eval_grid, y=upper_y,
                        mode='lines',
                        line=dict(width=0),
                        showlegend=False,
                        hoverinfo='skip'
                    ),
                    row=1, col=1
                )
                
                # Lower bound with fill
                fig.add_trace(
                    go.Scatter(
                        x=eval_grid, y=lower_y,
                        mode='lines',
                        line=dict(width=0),
                        fill='tonexty',
                        fillcolor=rgba_color,
                        showlegend=False,
                        hoverinfo='skip'
                    ),
                    row=1, col=1
                )
                
                # Mean trajectory line
                fig.add_trace(
                    go.Scatter(
                        x=eval_grid, y=mean_y,
                        mode='lines',
                        name=algo_name,
                        line=dict(color=color, width=2.5),
                        hovertemplate='Evals: %{x:.0f}<br>Error: %{y:.3e}'
                    ),
                    row=1, col=1
                )
                
            # --- Subplot 2: ECDF over target thresholds ---
            targets = np.logspace(-8, 2, 100)
            for algo_name, runs in algo_runs.items():
                color = ALGO_COLORS.get(algo_name, '#7f7f7f')
                hit_rates = []
                for t in targets:
                    hits = sum(1 for df in runs if np.min(df['raw_y'].values) <= t)
                    hit_rates.append(hits / len(runs))
                    
                fig.add_trace(
                    go.Scatter(
                        x=targets, y=hit_rates,
                        mode='lines',
                        name=algo_name,
                        line=dict(color=color, width=2.5),
                        showlegend=False,
                        hovertemplate='Target τ: %{x:.2e}<br>Fraction: %{y:.2f}'
                    ),
                    row=1, col=2
                )
                
            # Axis formatting
            fig.update_xaxes(type='log', title_text='Function Evaluations', row=1, col=1, gridcolor='#EBEBEB')
            fig.update_yaxes(type='log', title_text='Clean Distance to Optimum Δf(x)', row=1, col=1, gridcolor='#EBEBEB')
            
            fig.update_xaxes(type='log', title_text='Target Precision τ', row=1, col=2, gridcolor='#EBEBEB')
            fig.update_yaxes(title_text='ECDF (Fraction of Successful Trials)', range=[0, 1.05], row=1, col=2, gridcolor='#EBEBEB')
            
            fig.update_layout(
                template='plotly_white',
                font=dict(family='sans-serif', size=12),
                legend=dict(
                    orientation='h',
                    yanchor='bottom',
                    y=1.12,
                    xanchor='center',
                    x=0.5,
                    bgcolor='rgba(255,255,255,0.8)',
                    bordercolor='#E0E0E0',
                    borderwidth=1
                ),
                margin=dict(t=90, b=50, l=60, r=40),
                height=480,
                width=1000
            )
            
            out_fig_dir = FIGURES_DIR / f'{dim}D' / f'std_{noise_std}'
            out_fig_dir.mkdir(parents=True, exist_ok=True)
            
            png_path = out_fig_dir / f'f{p_id}.png'
            fig.write_image(str(png_path), scale=2)
            print(f'Saved Plotly PNG figure to {png_path}')


Saved Plotly PNG figure to /Users/nicolaibrahim/Desktop/proj/AAD_LLM/figures/2D/std_0.0/f1.png
Saved Plotly PNG figure to /Users/nicolaibrahim/Desktop/proj/AAD_LLM/figures/2D/std_0.0/f8.png
Saved Plotly PNG figure to /Users/nicolaibrahim/Desktop/proj/AAD_LLM/figures/2D/std_0.0/f11.png
Saved Plotly PNG figure to /Users/nicolaibrahim/Desktop/proj/AAD_LLM/figures/2D/std_0.0/f15.png
Saved Plotly PNG figure to /Users/nicolaibrahim/Desktop/proj/AAD_LLM/figures/2D/std_0.0/f21.png
Saved Plotly PNG figure to /Users/nicolaibrahim/Desktop/proj/AAD_LLM/figures/2D/std_0.05/f1.png
Saved Plotly PNG figure to /Users/nicolaibrahim/Desktop/proj/AAD_LLM/figures/2D/std_0.05/f8.png
Saved Plotly PNG figure to /Users/nicolaibrahim/Desktop/proj/AAD_LLM/figures/2D/std_0.05/f11.png
Saved Plotly PNG figure to /Users/nicolaibrahim/Desktop/proj/AAD_LLM/figures/2D/std_0.05/f15.png
Saved Plotly PNG figure to /Users/nicolaibrahim/Desktop/proj/AAD_LLM/figures/2D/std_0.05/f21.png
Saved Plotly PNG figure to /Users/nicol